# Amazon Voices — Phonetic Analysis

This notebook reproduces the three figures reported in the paper describing the **Amazon Voices** database.
It requires the 27 `.xlsx` concept files to be placed in the same directory as this notebook (or adjust the path in the *Data loading* section).

**Dependencies:** `pandas`, `numpy`, `matplotlib`, `openpyxl`

Install with:
```bash
pip install pandas numpy matplotlib openpyxl
```

In [ ]:
import pandas as pd
import unicodedata
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. IPA Tokenizer
Handles affricates (e.g. /tʃ/, /ts/), diacritics, and modifier letters.

In [ ]:
def tokenize_ipa(word):
    """Tokenize an IPA string into individual phones."""
    word = unicodedata.normalize('NFC', str(word).strip())
    tokens = []
    i = 0
    affricate_patterns = ['tʃ', 'dʒ', 'tɕ', 'dʑ', 'tʂ', 'dʐ', 'ts', 'dz', 'tɬ', 'dɮ']

    def is_combining(ch):
        return unicodedata.category(ch).startswith('M') or (0x0300 <= ord(ch) <= 0x036F)

    def is_modifier(ch):
        return (0x02B0 <= ord(ch) <= 0x02FF) or unicodedata.category(ch) == 'Lm'

    while i < len(word):
        if word[i] in ' -.,\n\t':
            i += 1
            continue
        matched = None
        for aff in affricate_patterns:
            if word[i:i+len(aff)] == aff:
                matched = aff
                break
        if matched:
            token = matched
            i += len(matched)
        else:
            token = word[i]
            i += 1
        while i < len(word) and (is_combining(word[i]) or is_modifier(word[i])):
            token += word[i]
            i += 1
        tokens.append(token)
    return tokens


BASE_VOWELS = set('aeiouɛɔɪʊæɑɒøœɯɨɘəɜɞʌʏ')
NASAL_VOWELS = set('ãẽĩõũ')

def is_vowel(phone):
    base = phone[0] if phone else ''
    return base in BASE_VOWELS or base in NASAL_VOWELS

## 2. Language and Family Mappings
Adjust `DATA_PATH` if your files are in a different folder.

In [ ]:
DATA_PATH = './'  # folder containing the .xlsx files

file_lang_map = {
    'amahuaca_concepts.xlsx':                ('Amahuaca',               'Segment'),
    'ashaninka_concepts.xlsx':               ('Ashaninka',              'segment'),
    'awajun_concepts.xlsx':                  ('Awajun',                 'Segment'),
    'bora_concepts.xlsx':                    ('Bora',                   'Segment'),
    'chaninawa_concepts.xlsx':               ('Chaninawa',              'Segment'),
    'huni_kuin_concepts.xlsx':               ('Huni Kuin',              'Segment'),
    'iskonawa_concepts.xlsx':                ('Iskonawa',               'segment'),
    'kakataibo (Sinchi Roca)_Concepts.xlsx': ('Kakataibo (Sinchi Roca)','Segment'),
    'kakataibo(aguaytia)_concepts.xlsx':     ('Kakataibo (Aguaytia)',   'Segment'),
    'madija_concepts.xlsx':                  ('Madija',                 'segment'),
    'marinawa_concepts.xlsx':                ('Marinawa',               'Segment'),
    'mastanawa_concepts.xlsx':               ('Mastanawa',              'Segment'),
    'matses_concepts.xlsx':                  ('Matses',                 'Segment'),
    'matsigenka_concepts .xlsx':             ('Matsigenka',             'segment'),
    'murui_concepts.xlsx':                   ('Murui-Muinanɨ',          'Segment'),
    'ocaina_concepts.xlsx':                  ('Ocaina',                 'Segment'),
    'pastazakichwa_concepts.xlsx':           ('Pastaza Kichwa',         'Segment'),
    'sharanahua_concepts.xlsx':              ('Sharanahua',             'Segment'),
    'shawandawa_concepts.xlsx':              ('Shawandawa',             'Segment'),
    'shawi_concepts.xlsx':                   ('Shawi',                  'Segment'),
    'shipibo_concepts.xlsx':                 ('Shipibo-Conibo',         'Segment'),
    'shiwilu_concepts.xlsx':                 ('Shiwilu',                'Segment'),
    'ticuna_concepts.xlsx':                  ('Ticuna',                 'segment'),
    'wampis_concepts.xlsx':                  ('Wampis',                 'Segment'),
    'yagua_concepts.xlsx':                   ('Yagua',                  'Segment'),
    'yanesha_concepts.xlsx':                 ('Yanesha',                'Segment'),
    'yine_concepts.xlsx':                    ('Yine',                   'Segment'),
}

family_map = {
    'Amahuaca': 'Pano-Tacanan',        'Ashaninka': 'Arawakan',
    'Awajun': 'Chicham',               'Bora': 'Boran',
    'Huni Kuin': 'Pano-Tacanan',       'Chaninawa': 'Pano-Tacanan',
    'Iskonawa': 'Pano-Tacanan',        'Kakataibo (Aguaytia)': 'Pano-Tacanan',
    'Kakataibo (Sinchi Roca)': 'Pano-Tacanan', 'Madija': 'Arawan',
    'Marinawa': 'Pano-Tacanan',        'Mastanawa': 'Pano-Tacanan',
    'Matses': 'Pano-Tacanan',          'Matsigenka': 'Arawakan',
    'Murui-Muinanɨ': 'Huitotoan',      'Ocaina': 'Huitotoan',
    'Pastaza Kichwa': 'Quechuan',      'Sharanahua': 'Pano-Tacanan',
    'Shawandawa': 'Pano-Tacanan',      'Shawi': 'Cahuapanan',
    'Shipibo-Conibo': 'Pano-Tacanan',  'Shiwilu': 'Cahuapanan',
    'Ticuna': 'Ticuna-Yuri',           'Wampis': 'Chicham',
    'Yagua': 'Peba-Yagua',             'Yanesha': 'Arawakan',
    'Yine': 'Arawakan',
}

family_colors = {
    'Pano-Tacanan': '#C0623A', 'Arawakan': '#3A7D6E', 'Chicham': '#5B5EA6',
    'Boran': '#B8860B',        'Arawan': '#8B2020',   'Cahuapanan': '#3A7A3A',
    'Huitotoan': '#B07D2A',    'Quechuan': '#2A7A7A', 'Peba-Yagua': '#6A3D8F',
    'Ticuna-Yuri': '#5A5A5A',
}

## 3. Data Processing
Reads each file, tokenizes the `Segment` column, and computes per-language statistics.

In [ ]:
results = {}
global_counter = Counter()
lang_phone_sets = {}

for fname, (lang_name, seg_col) in file_lang_map.items():
    try:
        df = pd.read_excel(DATA_PATH + fname)
        segments = df[seg_col].dropna().astype(str).tolist()
        segments = [s for s in segments if s.strip() and s.strip() != 'nan']
        all_phones = [phone for seg in segments for phone in tokenize_ipa(seg)]
        phone_counter = Counter(all_phones)
        global_counter.update(phone_counter)
        lang_phone_sets[lang_name] = set(phone_counter.keys())
        total_tokens = sum(phone_counter.values())
        vowel_tokens = sum(c for p, c in phone_counter.items() if is_vowel(p))
        cons_tokens  = total_tokens - vowel_tokens
        results[lang_name] = {
            'unique_phones': len(phone_counter),
            'cv_ratio': cons_tokens / vowel_tokens if vowel_tokens else None,
            'family': family_map.get(lang_name, 'Other'),
        }
    except Exception as e:
        print(f'Error processing {lang_name}: {e}')

print(f'Languages processed: {len(results)}')

## 4. Figures
### Shared style settings

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],
    'axes.facecolor': '#FAFAFA',
    'figure.facecolor': 'white',
    'axes.grid': False,
})

def apply_style(ax):
    ax.yaxis.grid(True, color='#E0E0E0', linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_color('#CCCCCC')
    ax.spines['bottom'].set_linewidth(0.8)
    ax.tick_params(axis='both', length=0, labelcolor='#333333')

### Figure 1 — Phonetic Inventory Size by Language

In [ ]:
langs_sorted = sorted(results.items(), key=lambda x: x[1]['unique_phones'], reverse=True)
lang_names = [l for l, _ in langs_sorted]
inv_sizes  = [d['unique_phones'] for _, d in langs_sorted]
bar_colors = [family_colors[d['family']] for _, d in langs_sorted]
families   = [d['family'] for _, d in langs_sorted]
mean_inv   = np.mean(inv_sizes)

fig, ax = plt.subplots(figsize=(15, 6.5))
apply_style(ax)

bars = ax.bar(range(len(lang_names)), inv_sizes, color=bar_colors,
              edgecolor='white', linewidth=0.8, width=0.72, zorder=3)

ax.axhline(mean_inv, color='#444444', linestyle='--', linewidth=1.2, zorder=4, alpha=0.7)
ax.text(len(lang_names) - 0.5, mean_inv + 1.5, f'mean = {mean_inv:.1f}',
        ha='right', va='bottom', fontsize=8.5, color='#444444', style='italic')

for bar, val in zip(bars, inv_sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.7,
            str(val), ha='center', va='bottom', fontsize=7.5,
            color='#222222', fontweight='bold')

families_present = sorted(set(families))
legend_patches = [mpatches.Patch(color=family_colors[f], label=f) for f in families_present]
legend_patches.append(plt.Line2D([0],[0], color='#444444', linestyle='--', lw=1.2,
                                  label=f'Mean ({mean_inv:.1f})'))
ax.legend(handles=legend_patches, fontsize=8.5, loc='upper right', ncol=2,
          framealpha=0.85, edgecolor='#CCCCCC')

ax.set_xticks(range(len(lang_names)))
ax.set_xticklabels(lang_names, rotation=40, ha='right', fontsize=9.5)
ax.set_ylabel('Number of distinct phones', fontsize=11, labelpad=8)
ax.set_ylim(0, max(inv_sizes) + 14)

plt.tight_layout()
plt.savefig('fig1_inventory_size.png', dpi=180, bbox_inches='tight')
plt.show()

### Figure 2 — Cross-Linguistic Phone Coverage (≥89%)

In [ ]:
n_langs = len(results)
phone_lang_count = {phone: sum(1 for s in lang_phone_sets.values() if phone in s)
                    for phone in global_counter}

coverage_df = pd.DataFrame([
    {'phone': p, 'n_langs': c, 'pct': c / n_langs * 100,
     'total_tokens': global_counter[p], 'is_vowel': is_vowel(p)}
    for p, c in phone_lang_count.items()
]).sort_values('n_langs', ascending=False)

plot_df = coverage_df[coverage_df['pct'] >= 89].sort_values('n_langs', ascending=False)

vowel_color = '#3A7D6E'
cons_color  = '#C0623A'
bar_cols2 = [vowel_color if v else cons_color for v in plot_df['is_vowel']]

fig2, ax2 = plt.subplots(figsize=(10, 5.5))
apply_style(ax2)

bars2 = ax2.bar(range(len(plot_df)), plot_df['n_langs'], color=bar_cols2,
                edgecolor='white', linewidth=0.8, width=0.65, zorder=3)

for n_val, label, ls in [(27, 'Universal (100%)', '-'), (24, '≥89%', '--')]:
    ax2.axhline(n_val, color='#555555', linestyle=ls, linewidth=1.0, alpha=0.6)
    ax2.text(len(plot_df) - 0.4, n_val + 0.2, label,
             ha='right', va='bottom', fontsize=7.5, color='#555555', style='italic')

for bar, val in zip(bars2, plot_df['n_langs']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             str(val), ha='center', va='bottom', fontsize=8,
             color='#222222', fontweight='bold')

ax2.set_xticks(range(len(plot_df)))
ax2.set_xticklabels(plot_df['phone'], rotation=0, fontsize=12)
ax2.set_ylabel('Number of languages', fontsize=11, labelpad=8)
ax2.set_ylim(0, n_langs + 4)

legend_patches2 = [
    mpatches.Patch(color=vowel_color, label='Vowel'),
    mpatches.Patch(color=cons_color, label='Consonant'),
]
ax2.legend(handles=legend_patches2, fontsize=9, loc='lower left',
           framealpha=0.85, edgecolor='#CCCCCC')

plt.tight_layout()
plt.savefig('fig2_phone_coverage.png', dpi=180, bbox_inches='tight')
plt.show()

### Figure 3 — Consonant-to-Vowel Ratio by Language

In [ ]:
cv_sorted = sorted(
    [(l, d) for l, d in results.items() if d['cv_ratio'] is not None],
    key=lambda x: x[1]['cv_ratio'], reverse=True
)
cv_langs    = [l for l, _ in cv_sorted]
cv_vals     = [d['cv_ratio'] for _, d in cv_sorted]
cv_colors   = [family_colors[d['family']] for _, d in cv_sorted]
cv_families = [d['family'] for _, d in cv_sorted]
mean_cv     = np.mean(cv_vals)

fig3, ax3 = plt.subplots(figsize=(15, 6.5))
apply_style(ax3)

bars3 = ax3.bar(range(len(cv_langs)), cv_vals, color=cv_colors,
                edgecolor='white', linewidth=0.8, width=0.72, zorder=3)

ax3.axhline(mean_cv, color='#444444', linestyle='--', linewidth=1.2, zorder=4, alpha=0.7)
ax3.axhline(1.0, color='#888888', linestyle=':', linewidth=1.0, zorder=4, alpha=0.6)
ax3.text(len(cv_langs) - 0.5, mean_cv + 0.01, f'mean = {mean_cv:.2f}',
         ha='right', va='bottom', fontsize=8.5, color='#444444', style='italic')
ax3.text(len(cv_langs) - 0.5, 1.0 + 0.01, 'C/V = 1.0',
         ha='right', va='bottom', fontsize=8.0, color='#888888', style='italic')

for bar, val in zip(bars3, cv_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.2f}', ha='center', va='bottom', fontsize=7.0,
             color='#222222', fontweight='bold')

families_present3 = sorted(set(cv_families))
legend_patches3 = [mpatches.Patch(color=family_colors[f], label=f) for f in families_present3]
legend_patches3.append(plt.Line2D([0],[0], color='#444444', linestyle='--', lw=1.2,
                                   label=f'Mean ({mean_cv:.2f})'))
ax3.legend(handles=legend_patches3, fontsize=8.5, loc='upper right', ncol=2,
           framealpha=0.85, edgecolor='#CCCCCC')

ax3.set_xticks(range(len(cv_langs)))
ax3.set_xticklabels(cv_langs, rotation=40, ha='right', fontsize=9.5)
ax3.set_ylabel('Consonant-to-Vowel ratio', fontsize=11, labelpad=8)
ax3.set_ylim(0, max(cv_vals) + 0.15)

plt.tight_layout()
plt.savefig('fig3_cv_ratio.png', dpi=180, bbox_inches='tight')
plt.show()